# Simple MCP demo
- Send a query to LLM, says it doesn't know
- Give it a tool to help, and it knows!

- MCP has 3 components:
  - An MCP client which manages convo (like a chat client or an IDE that is asking an LLM for help with code)
  - An MCP server which provides tools for a purpose, like looking up documents in a vector DB
  - An LLM which supports tool use

- Flow:
  - MCP client connects to MCP server(s)
  - MCP server can list the tools it offers to the clinet
  - Client can prompt the LLM, providing a list of available tools (calling signatures, and semantic descriptions of when to call them)
  - LLM attempts to respond. If, based on the prompt and the available tools, a tool would be the best way to answer the question, LLM will respond with a calling signature
  - Client then calls the tools using the provided signature, adds the output from the tool to the conversation, and calls the LLM again with the updated conversation
  - LLM may respond with further tool call requests, or provide a response!
  - That's mostly it, besides executing tool calls, servers can also provide static resources for the client like docs, reference prompts, see the [docs on the home page](https://modelcontextprotocol.io/overview)


- MCP is like a USB standard for LLM/tool integration


- &gt; 10,000 MCP servers available 
  - [MCP Market leaderboard (by GitHub stars)](https://mcpmarket.com/leaderboards)
  - [PulseMCP directory (by downloads)](https://www.pulsemcp.com/servers?sort=popular-30-days-desc)
  - [LobeHub](https://lobehub.com/mcp)
  - [Glama](https://glama.ai/mcp/servers)


- More info:
  - [Anthropic MCP Announcement](https://www.anthropic.com/news/model-context-protocol)
  - [Anthropic YouTube talk](https://www.youtube.com/watch?v=kQmXtrmQ5Zg)
  - [Model Context Protocol home page on GitHub](https://github.com/modelcontextprotocol)
  - [Composio intro](https://composio.dev/blog/what-is-model-context-protocol-mcp-explained)
      
      
  

![!image.png](q5AltSX5E3TfLsmtZp5jjLMU5U.png)


In [12]:
import sys
import os
import dotenv
import re
from datetime import datetime, timedelta
import time
from typing import Dict, Any, Optional, Annotated
from urllib.parse import urljoin, urlparse

import asyncio
import nest_asyncio

from contextlib import AsyncExitStack
import mcp
from mcp.client.stdio import stdio_client
from mcp import ClientSession, StdioServerParameters

import anthropic
from anthropic import Anthropic
import pdb


In [3]:
# load secrets from .env
dotenv.load_dotenv()

# enable asyncio in jupyter notebook
nest_asyncio.apply()

# Initialize plotly for Jupyter
# init_notebook_mode(connected=True)


In [5]:
client = anthropic.Anthropic()

# https://docs.anthropic.com/en/docs/about-claude/models/overview
claude_4_models = [
    "claude-opus-4-20250514",
    "claude-sonnet-4-20250514"
]

print("Available Claude 4 models:")
print("\n".join(claude_4_models))
print()

# Try making a simple completion request to each:

message = "what is the airspeed velocity of an unladen swallow"
for model in claude_4_models:
    try:
        response = client.messages.create(
            model=model,
            max_tokens=200,
            messages=[{"role": "user", "content": message}]
        )
        print(f"✓ {model}")
        print(response.content[0].text)
        print()
    except Exception as e:
        print(f"✗ {model} - error: {str(e)}")

Available Claude 4 models:
claude-opus-4-20250514
claude-sonnet-4-20250514

✓ claude-opus-4-20250514
Are you referring to an African or European swallow?

(This is, of course, a reference to the famous scene from "Monty Python and the Holy Grail"!)

But to give you a real answer: European swallows have been estimated to fly at about 20.1 miles per hour (32.4 km/h) during normal flight, though this can vary based on wind conditions and whether they're migrating. African swallows would have different flight characteristics, but there's less specific data available for them.

✓ claude-sonnet-4-20250514
Ah, a classic Monty Python reference! 

But you forgot to ask: "What do you mean? An African or European swallow?"

In the spirit of the Holy Grail, here are some actual numbers:

**European Swallow (Barn Swallow):**
- Cruising speed: ~17-20 mph (27-32 km/h)
- Maximum speed: ~35-60 mph (56-97 km/h)

**African Swallow (Red-rumped Swallow):**
- Similar speeds to European swallows
- Cruising: 

In [28]:
class MCPClient:
    """An MCP client adapted to run in a Jupyter notebook.
    """
    def __init__(self):
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.anthropic = Anthropic()
        self.tools = {}
        self.tools_reverse = {}

    def connect_to_server(self, server_script_path: str):
        """Connect to an MCP server and list its tools."""
        print(f"Connecting to server: {server_script_path}...")
        is_python = server_script_path.endswith('.py')
        if not is_python:
            raise ValueError("Server script must be a .py file")

        server_params = StdioServerParameters(
            command=sys.executable,  # Use the same python executable
            args=[server_script_path],
            env=None
        )

        pdb.set_trace()
        # print(server_params)
        response = asyncio.run(self.async_connect_to_server(server_params))
        # print(response)
        self.tools[server_script_path] = response.tools
        reverse_tool_dict = {tool.name: server_script_path for tool in response.tools}
        self.tools_reverse = {**self.tools_reverse, **reverse_tool_dict}
        print("\nConnection successful!")
        print("Available tools:", [tool.name for tool in self.tools[server_script_path]])

    async def async_connect_to_server(self, server_params):
        
        stdio_transport = await self.exit_stack.enter_async_context(stdio_client(server_params))
        self.stdio, self.write = stdio_transport
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))

        await self.session.initialize()
        response = await self.session.list_tools()
        return response
    

    def process_query(self, query: str) -> str:
        """Process a query using LLM and the available tools."""
        if not self.session:
            return "Error: Not connected to a server. Please run connect_to_server first."

        pdb.set_trace()
        messages = [{"role": "user", "content": query}]
        available_tools = [{
            "name": tool.name,
            "description": tool.description,
            "input_schema": tool.inputSchema
        } for server in self.tools.values() for tool in server]
        print("Sending query to LLM...")
        response = self.anthropic.messages.create(
            model="claude-sonnet-4-20250514", 
            max_tokens=1024,
            messages=messages,
            tools=available_tools
        )

        final_text = []
        for content in response.content:
            if content.type == 'text':
                final_text.append(content.text)
            elif content.type == 'tool_use':
                tool_name = content.name
                tool_args = content.input
                print(f"Claude requested to use tool: {tool_name} with arguments: {tool_args}")

                result = asyncio.run(self.session.call_tool(tool_name, tool_args))
                print("Received tool result from server.")

                # Create the tool result content block
                tool_result_content = {
                    "type": "tool_result",
                    "tool_use_id": content.id,
                    "content": str(result.content) # Ensure content is a string
                }

                # Append the original assistant message and the tool result
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user", "content": [tool_result_content]})

                # Get next response from Claude
                print(f"Tool result: {tool_result_content}")
                print("Sending tool result back to Claude...")
                follow_up_response = self.anthropic.messages.create(
                    model="claude-sonnet-4-20250514",
                    max_tokens=1024,
                    messages=messages,
                )
                for follow_up_content in follow_up_response.content:
                    if follow_up_content.type == 'text':
                        final_text.append(follow_up_content.text)

        return "\n".join(final_text)
    
    def chat_loop(self):
        """Run an interactive chat loop"""
        print("\nMCP Client Started!")
        print("Type your queries or 'quit' to exit.")

        while True:
            try:
                query = input("\nQuery: ").strip()

                if query.lower() == 'quit':
                    break

                response = self.process_query(query)
                print("\n" + response)

            except Exception as e:
                print(f"\nError: {str(e)}")
                
    def cleanup(self):
        """Clean up resources and close the server connection."""
        print("Cleaning up resources...")
        asyncio.run(self.exit_stack.aclose())
        print("Cleanup complete.")


In [29]:
def connect():
    client = MCPClient()
    client.connect_to_server('swallow_server.py')
    return client

# Run the connection and keep the client object
# This will block until the connection is established.
client = connect()


Connecting to server: swallow_server.py...
> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_10551/2653960807.py(26)connect_to_server()
     24         pdb.set_trace()
     25         # print(server_params)
---> 26         response = asyncio.run(self.async_connect_to_server(server_params))
     27         # print(response)
     28         self.tools[server_script_path] = response.tools

ipdb> c

Connection successful!
Available tools: ['unladen_swallow_airspeed']


In [30]:
def run_query(query):
    response = client.process_query(query)
    print("\n--- Claude's Response ---")
    print(response)
    print("-------------------------")


In [31]:
# Run a query
query = "What is the airspeed velocity of an unladen European swallow"
run_query(query)


> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_10551/2653960807.py(51)process_query()
     49 
     50         pdb.set_trace()
---> 51         messages = [{"role": "user", "content": query}]
     52         available_tools = [{
     53             "name": tool.name,

ipdb> c
Sending query to LLM...
Claude requested to use tool: unladen_swallow_airspeed with arguments: {'swallow_type': 'european'}
Received tool result from server.
Tool result: {'type': 'tool_result', 'tool_use_id': 'toolu_01HmcxMJkrR75hZ1uqF5ENRN', 'content': "[TextContent(type='text', text='27.1828km/h', annotations=None, meta=None)]"}
Sending tool result back to Claude...

--- Claude's Response ---
An unladen European swallow has an airspeed velocity of approximately 27.18 km/h (about 16.9 mph or 7.5 m/s).

*But of course, if you're referencing Monty Python and the Holy Grail, the real answer is that this is the question the Bridge Keeper should have asked first, rather than asking about African swallows

In [32]:
client.chat_loop()


MCP Client Started!
Type your queries or 'quit' to exit.
ERROR! Session/line number was not unique in database. History logging moved to new session 2186

Query: what is the airspeed velocity of an unladen swallow?
> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_10551/2653960807.py(51)process_query()
     49 
     50         pdb.set_trace()
---> 51         messages = [{"role": "user", "content": query}]
     52         available_tools = [{
     53             "name": tool.name,

ipdb> c
Sending query to LLM...

I can help you find the airspeed velocity of an unladen swallow! However, I need to know which type of swallow you're asking about - African or European?

This is a classic reference to Monty Python and the Holy Grail, where this very question is asked at the Bridge of Death. In the movie, the bridge keeper asks this question, and the correct response is to ask "What do you mean? An African or European swallow?"

Which type would you like me to look up for you?

Qu